# Upstream NanoBEIR: establish correctness first

Use a **fresh GPU runtime**, then run all cells. This notebook checks ModernColBERT against its published NanoBEIR scores using upstream PyLate scoring and official qrels. It then compares dense dot product, ColBERT, cross-encoder and BM25 on the same candidates.

All 13 datasets are selected by default. No CLIP labels, metadata filters, MUVERA or custom MaxSim. The first run is a reproduction attempt; it does not assume the published scores will match.


In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

def run_logged(command, *, cwd=None, log_path):
    """Forward child stdout/stderr through notebook output and keep the failure tail."""
    import collections
    import subprocess
    import sys
    from pathlib import Path

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = collections.deque(maxlen=80)
    print(f'Python: {sys.version.split()[0]} | executable: {sys.executable}', flush=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n--- New invocation ---\n')
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                              errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    tail.append(line)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=5)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if returncode:
        raise RuntimeError(
            f'Command exited with status {returncode}. Full log: {log_path}\n'
            + ''.join(tail))
    return returncode

drive.mount('/content/drive')
REPO = Path('/content/ras-upstream-nanobeir')
BRANCH = 'codex/colbert-muvera-baselines'
LOGS = Path('/content/drive/MyDrive/ras_upstream_nanobeir_logs')
if not REPO.exists():
    run_logged(['git', 'clone', '--branch', BRANCH, '--single-branch',
                'https://github.com/hanialshater/ras.git', str(REPO)],
               log_path=LOGS / 'setup.log')
else:
    run_logged(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH],
               log_path=LOGS / 'setup.log')
print(subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True))
run_logged([sys.executable, '-m', 'pip', 'install', '-e',
            str(REPO) + '[dev,upstream-retrieval]'],
           log_path=LOGS / 'install.log')


In [ ]:
import os
os.chdir(REPO)
os.environ['PYTHONPATH'] = str(REPO / 'src') + ':' + str(REPO)
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['PYLATE_SCORES_BACKEND'] = 'torch'
# Imports must succeed so the native integration test cannot silently skip.
run_logged([sys.executable, '-c', 'import torch, pylate, sentence_transformers; assert torch.cuda.is_available(), "Select a GPU runtime"'], cwd=REPO, log_path=LOGS / 'environment.log')
run_logged([sys.executable, '-m', 'pytest', '-q', 'tests/test_upstream_nanobeir.py'], cwd=REPO, log_path=LOGS / 'tests.log')


## 1. Reproduce the reference

Models and datasets are pinned to immutable revisions; the actual text, judgments, configuration and all retrieved scores are saved. The published mean **0.6758** applies to all 13 datasets. A per-dataset absolute deviation above **0.01** triggers investigation before the comparison; this is not a statistical equivalence threshold.

For a shorter smoke run set `DATASETS = ['scifact', 'nfcorpus']`. Use a new `RUN` directory if changing package versions, scoring chunk size or source. Completed datasets resume automatically.


In [ ]:
RUN = Path('/content/drive/MyDrive/ras_upstream_nanobeir_seed7_v1')
DATASETS = ['climatefever', 'dbpedia', 'fever', 'fiqa2018', 'hotpotqa', 'msmarco', 'nfcorpus', 'nq', 'quoraretrieval', 'scidocs', 'arguana', 'scifact', 'touche2020']
BASE = [sys.executable, '-u', '-m', 'experiments.upstream_nanobeir', '--output-dir', str(RUN), '--datasets', *DATASETS]
run_logged(BASE + ['--phase', 'reference'], cwd=REPO, log_path=LOGS / 'reference.log')


In [ ]:
import json
import pandas as pd
from IPython.display import display
reference = pd.read_csv(RUN / 'reference_check.csv')
display(reference[['dataset', 'queries', 'published', 'observed', 'difference', 'within_0_01']])
print(json.dumps(json.loads((RUN / 'reference_summary.json').read_text()), indent=2))
print('Actual ColBERT settings:')
print((RUN / 'colbert_model.json').read_text())


## 2. Compare the same candidates

Each method scores the identical dense top-100 pool, with full NanoBEIR qrels as the relevance denominator. Candidate coverage is reported separately. Dense, ColBERT and BM25 also have full-corpus retrieval results. Dense uses normalized dot product (upstream cosine). BM25 is a simple rank-bm25 lexical control, not Lucene reproduction.

If the reference check fails, the default cell leaves comparison pending and still lets the export cell run. Send the reference table and ZIP for diagnosis. Set `ALLOW_REFERENCE_MISMATCH = True` only to run an explicitly labeled diagnostic comparison.


In [ ]:
POOL_SIZE = 100
ALLOW_REFERENCE_MISMATCH = False
COMPARE = RUN / f'comparison_pool{POOL_SIZE}'
REFERENCE_OK = bool(reference.within_0_01.all())
if REFERENCE_OK or ALLOW_REFERENCE_MISMATCH:
    flags = ['--allow-reference-mismatch'] if ALLOW_REFERENCE_MISMATCH else []
    run_logged(BASE + ['--phase', 'compare', '--pool-size', str(POOL_SIZE)] + flags, cwd=REPO, log_path=LOGS / 'comparison.log')
else:
    print('Reference differs from the card. Comparison pending; export the reference results below.')


In [ ]:
if (COMPARE / 'quality.csv').exists():
    quality = pd.read_csv(COMPARE / 'quality.csv')
    print('Dataset macro means; full-corpus and shared-pool scopes kept separate')
    display(quality.groupby(['scope', 'method'])[['ndcg10', 'mrr10', 'recall10']].mean().reset_index())
    print('Per-dataset results')
    display(quality)
    print('Paired nDCG@10 differences: method A minus method B, per dataset')
    display(pd.read_csv(COMPARE / 'paired_ndcg.csv'))
    print('Relevant-document candidate coverage')
    display(pd.read_csv(COMPARE / 'pool_coverage.csv').groupby('dataset').relevant_coverage.agg(['mean', 'min']))
    print('Model settings and reference status')
    print((COMPARE / 'models.json').read_text())
else:
    print('Comparison has not run; reference results remain available.')


## Export

The ZIP contains dataset snapshots, pinned settings, upstream predictions, comparison tables and logs. Offline job durations are **not serving latency**. Matching backbone size does not match supervision, context limits or compute. Assess the reference first; MUVERA and serving costs follow separately.


In [ ]:
import zipfile
from google.colab import files
archive = RUN.parent / (RUN.name + '.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in RUN.rglob('*'):
        if path.is_file():
            z.write(path, str(path.relative_to(RUN)))
    for path in LOGS.glob('*.log'):
        z.write(path, 'logs/' + path.name)
print(archive)
files.download(str(archive))
